In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os

## 1. Import Required Libraries

# Collaborative Filtering Recommendation System

This notebook implements a collaborative filtering recommendation system using user-item interactions.

In [1]:
import pandas as pd
import numpy as np

## Load Dataset

In [2]:
 df = pd.read_csv(r"E:\Internship\zatch-reel-recommender(1)\data\reel_recommendation_dataset_1000_rows.csv")

In [3]:
df

,user_id,reel_id,watch_time,watch_percentage,liked,shared,saved,clicked,purchased,category,brand,timestamp
0,103,424,20,9,1,0,0,1,1,Home,Titan,2024-10-17 18:32:00
1,180,369,38,87,1,0,1,1,1,Fashion,Titan,2024-06-25 20:00:00
2,93,236,4,38,1,0,0,0,0,Beauty,Zara,2024-09-27 08:16:00
3,15,83,16,14,0,0,0,0,0,Sports,Titan,2024-04-22 12:08:00
4,107,42,15,93,0,0,0,1,0,Shoes,Apple,2024-02-24 18:18:00
...,...,...,...,...,...,...,...,...,...,...,...,...
995,111,363,45,83,1,0,0,1,1,Beauty,Boat,2024-01-23 06:29:00
996,111,437,4,45,0,0,0,0,1,Home,Nike,2024-08-26 19:25:00
997,34,158,25,67,0,1,1,0,0,Home,Boat,2024-10-19 01:07:00
998,111,39,21,27,1,1,1,1,1,Beauty,Puma,2024-08-08 20:41:00


In [4]:
df.shape

(1000, 12)

In [5]:
df.isnull().sum()

user_id             0
reel_id             0
watch_time          0
watch_percentage    0
liked               0
shared              0
saved               0
clicked             0
purchased           0
category            0
brand               0
timestamp           0
dtype: int64

In [6]:
df['interaction_score'] = (
    (df['watch_percentage'] * 0.1)
    + (df['liked'] * 3)
    + (df['shared'] * 5)
    + (df['saved'] * 4)
    + (df['clicked'] * 2)
    + (df['purchased'] * 10)
)

In [7]:
df[['user_id','reel_id','interaction_score']].head()

,user_id,reel_id,interaction_score
0,103,424,15.9
1,180,369,27.7
2,93,236,6.8
3,15,83,1.4
4,107,42,11.3


In [8]:
user_reel_matrix = df.pivot_table(
    index='user_id',
    columns='reel_id',
    values='interaction_score',
    fill_value=0
)

In [9]:
user_reel_matrix.shape

(198, 426)

## Build User Similarity Matrix

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
user_similarity = cosine_similarity(user_reel_matrix)

In [13]:
user_similarity.shape

(198, 198)

## Convert Similarity Matrix To DataFrame

In [14]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_reel_matrix.index,
    columns=user_reel_matrix.index
)

In [15]:
user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,191,192,193,194,195,196,197,198,199,200
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,0.0,0.185446,0.220285,0.160768,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.231194,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,1.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.289873,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
user_id = 103

In [17]:
user_similarity_df[user_id].sort_values(
    ascending=False
).head(10)

user_id
103    1.000000
98     0.374547
61     0.259103
147    0.221935
56     0.215602
19     0.164026
163    0.159666
17     0.138272
28     0.132787
37     0.132306
Name: 103, dtype: float64

In [18]:
user_similarity.shape

(198, 198)

In [19]:
user_similarity_df[user_id].sort_values(
    ascending=False
).head(10)

user_id
103    1.000000
98     0.374547
61     0.259103
147    0.221935
56     0.215602
19     0.164026
163    0.159666
17     0.138272
28     0.132787
37     0.132306
Name: 103, dtype: float64

In [22]:
def recommend_reels(user_id, n_recommendations=5):

    similar_users = user_similarity_df[user_id].sort_values(
        ascending=False
    )[1:6]

    recommended_reels = {}

    for similar_user, similarity_score in similar_users.items():

        reels_watched = user_reel_matrix.loc[similar_user]

        for reel_id, score in reels_watched.items():

            if score > 0:

                if reel_id not in recommended_reels:
                    recommended_reels[reel_id] = 0

                recommended_reels[reel_id] += score * similarity_score

    recommended_reels = sorted(
        recommended_reels.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return recommended_reels[:n_recommendations]

In [23]:
recommend_reels(103)

[(92, 10.44127837834528),
 (366, 9.101486369807798),
 (332, 8.709732426867891),
 (65, 7.453480607373463),
 (344, 6.458309098595841)]

In [24]:
import pickle

pickle.dump(user_similarity_df,
            open("user_similarity.pkl", "wb"))

pickle.dump(user_reel_matrix,
            open("user_reel_matrix.pkl", "wb"))